In [ ]:
# Midterm manure Q3 JuMP reference solution
using JuMP, HiGHS

P = ["DM"]
S = ["DF"]
D = ["CF", "SF"]
L = ["DM->CF", "DM->SF"]

sprod = Dict(zip(S, ["DM"]))
sub = Dict(zip(S, [1000]))
sbid = Dict(zip(S, [-0.7]))

dprod = Dict(zip(D, ["DM", "DM"]))
dub = Dict(zip(D, [500, 500]))
dbid = Dict(zip(D, [-0.5, 1.5]))

flocs = Dict(zip(L, ["DF", "DF"]))
flocr = Dict(zip(L, ["CF", "SF"]))
fub = Dict(zip(L, [1000, 1000]))
fprod = Dict(zip(L, ["DM", "DM"]))
fbid = Dict(zip(L, [0.1, 0.2]))

In [ ]:
m = Model(HiGHS.Optimizer)

@variable(m, s[S] >= 0)
@variable(m, d[D] >= 0)
@variable(m, f[L] >= 0)

@constraint(m, [i in S], s[i] <= sub[i])
@constraint(m, [j in D], d[j] <= dub[j])
@constraint(m, [l in L], f[l] <= fub[l])
@constraint(m, sbal[i in S], s[i] == sum(f[l] for l in L if flocs[l] == i && fprod[l] == sprod[i]))
@constraint(m, dbal[j in D], d[j] == sum(f[l] for l in L if flocr[l] == j && fprod[l] == dprod[j]))

dcost = @expression(m, sum(dbid[j] * d[j] for j in D))
scost = @expression(m, sum(sbid[i] * s[i] for i in S))
fcost = @expression(m, sum(fbid[l] * f[l] for l in L))
@objective(m, Max, dcost - scost - fcost)

optimize!(m)

In [ ]:
value.(s), value.(d), value.(f), value(dcost), value(scost), value(fcost), objective_value(m)